In [2]:
import pickle
import torch
import os
import numpy as np
import pandas as pd
from scipy import stats

def determine_better_direction(metric_name):
    better_is_higher = [
        'r_squared',
    ]

    if any(better_metric in metric_name.lower() for better_metric in better_is_higher):
        return 1
    else:
        return -1


model_name_config = {
    'biolord': '/home/usr/sc2Flow_lab/biolord/results',
    'chemCPA': '/home/usr/sc2Flow_lab/chemCPA/chemCPA/results/',
    'sc2Flow': '/home/usr/sc2Flow_lab/sc2Flow/output',
}


# data_store[dataset][metric][model_name] = [split1_score, split2_score, ...]
data_store = {}

models_list = [
    'chemCPA',
    'biolord',
    'sc2Flow']
my_method = 'sc2Flow'

print("reading data...")

for model_name in models_list:
    for dataset, splits in zip(['mcfarland', 'zhaoSims'], [5, 4]):
        for split_id in range(splits):
            if splits != 1:
                split_key = f'split_{split_id}'
            else:
                split_key = 'mode'

            # Since we didn't set a unified path configuration method, conditional handling is required.
            # We will improve this in future versions.
            if model_name in ['sc2Flow','CellFlow','unpredicted',]:
                res_path = f'{model_name_config[model_name]}/{dataset}/{split_key}/res.pkl'
                res_de_path = f'{model_name_config[model_name]}/{dataset}/{split_key}/res_de.pkl'
            elif model_name in ['biolord','chemCPA']:
                res_path = f'{model_name_config[model_name]}/{dataset}/{split_key}/res.pkl'
                res_de_path = f'{model_name_config[model_name]}/{dataset}/{split_key}/res_deg.pkl'
            else:
                raise ValueError(f"[Error] Unknown model: {model_name}")

            if not os.path.exists(res_path):
                print(f"[Warning] Path missing: {res_path}")
                continue
            if not os.path.exists(res_de_path):
                print(f"[Warning] Path missing: {res_de_path}")
                continue

            with open(res_path, 'rb') as f:
                result = pickle.load(f)
            with open(res_de_path, 'rb') as f:
                result_de = pickle.load(f)

            if dataset not in data_store:
                data_store[dataset] = {}

            # all genes
            temp_metrics = {}
            for c, metrics in result.items():
                for m, v in metrics.items():
                    key = f'test_{m}_mean'
                    if key not in temp_metrics:
                        temp_metrics[key] = []
                    temp_metrics[key].append(v)

            for metric, values in temp_metrics.items():
                if metric not in data_store[dataset]:
                    data_store[dataset][metric] = {m: [] for m in models_list}
                values_array = np.array([float(v) for v in values])
                non_nan_values = values_array[~np.isnan(values_array)]

                split_mean = float(torch.tensor(non_nan_values).mean().item())
                data_store[dataset][metric][model_name].append(split_mean)

            # DEGs
            temp_de_metrics = {}
            for c, metrics in result_de.items():
                for m, v in metrics.items():
                    if np.isnan(float(v)):
                        continue

                    key = f'test_{m}_mean_de'
                    if key not in temp_de_metrics:
                        temp_de_metrics[key] = []
                    temp_de_metrics[key].append(v)

            for metric, values in temp_de_metrics.items():
                if metric not in data_store[dataset]:
                    data_store[dataset][metric] = {m: [] for m in models_list}
                values_array = np.array([float(v) for v in values])
                non_nan_values = values_array[~np.isnan(values_array)]

                split_mean = float(torch.tensor(non_nan_values).mean().item())
                data_store[dataset][metric][model_name].append(split_mean)


print("data loading finished, transferring to DataFrame...")

# ==================== Results ====================

final_results_list = []

for dataset in data_store:
    print(f"Dataset: {dataset}")
    for metric in data_store[dataset]:
        models_data = data_store[dataset][metric]

        my_scores = models_data.get(my_method, [])

        if not my_scores:
            continue

        my_mean = np.mean(my_scores)
        my_std = np.std(my_scores, ddof=1) if len(my_scores) > 1 else 0.0

        row_data = {
            'Dataset': dataset,
            'Metric': metric,
            f'{my_method} (Mean±SD)': f"{my_mean:.4f} ± {my_std:.4f}"
        }

        for model in models_list:
            if model == my_method:
                continue

            other_scores = models_data.get(model, [])

            if not other_scores:
                row_data[f'{model} (Mean±SD)'] = "N/A"
                row_data[f'{model} p-val'] = "-"
                continue

            other_mean = np.mean(other_scores)
            other_std = np.std(other_scores, ddof=1) if len(other_scores) > 1 else 0.0

            # T-test
            better_direction = determine_better_direction(metric)

            if len(my_scores) > 1 and len(other_scores) > 1:
                t_stat, p_val = stats.ttest_ind(my_scores, other_scores, equal_var=False)
                p_str = f"{p_val:.3e}" if p_val < 0.001 else f"{p_val:.4f}"

                my_performance = my_mean * better_direction
                other_performance = other_mean * better_direction

                sig = ""
                if p_val < 0.05 and my_performance > other_performance:
                    sig = "*"
                elif p_val < 0.01 and my_performance > other_performance:
                    sig = "**"
                elif p_val < 0.05 and my_performance <= other_performance:
                    sig = "^"
                elif p_val < 0.01 and my_performance <= other_performance:
                    sig = "^^"

                p_display = f"{p_str}{sig}"
            else:
                p_display = "N/A (n=1)"

            row_data[f'{model} (Mean±SD)'] = f"{other_mean:.4f} ± {other_std:.4f}"
            row_data[f'{model} p-val'] = p_display

        final_results_list.append(row_data)

df = pd.DataFrame(final_results_list)

cols = ['Dataset', 'Metric'] + [c for c in df.columns if c not in ['Dataset', 'Metric']]
df = df[cols]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df)

def parse_mean_from_str(val_str):
    if not isinstance(val_str, str) or "±" not in val_str:
        return -np.inf
    try:
        return float(val_str.split('±')[0].strip())
    except:
        return -np.inf


def get_significance_mark(p_val_str):
    if not isinstance(p_val_str, str):
        return ""
    if "**" in p_val_str:
        return "^{**}}"
    elif "*" in p_val_str:
        return "^{*}"
    return ""


def generate_latex_tables(df, models_list, my_method):
    datasets = df['Dataset'].unique()

    model_display_names = {
        'chemCPA': 'chemCPA',
        'biolord': 'biolord',
        'sc2Flow': '\\textbf{sc2Flow} (Ours)'
    }

    print("\n" + "=" * 20 + " LaTeX Table " + "=" * 20 + "\n")

    for dataset in datasets:
        ds_df = df[df['Dataset'] == dataset].copy()

        latex_code = []
        latex_code.append(r"\begin{table}[htbp]")
        latex_code.append(r"\centering")
        latex_code.append(r"\caption{Performance comparison on \textbf{" + dataset + r"} dataset.}")
        latex_code.append(r"\label{tab:" + dataset + r"}")
        latex_code.append(r"\resizebox{\textwidth}{!}{%")

        col_format = "l" + "r" * len(models_list)
        latex_code.append(r"\begin{tabular}{" + col_format + r"}")
        latex_code.append(r"\toprule")

        header = ["Metric"] + [model_display_names.get(m, m) for m in models_list]
        latex_code.append(" & ".join(header) + r" \\")
        latex_code.append(r"\midrule")

        for _, row in ds_df.iterrows():
            metric_name = row['Metric']
            better_dir = determine_better_direction(metric_name)

            means = {}
            row_strings = {}

            for model in models_list:
                col_mean_std = f"{model} (Mean±SD)"
                col_pval = f"{model} p-val"

                val_str = row.get(col_mean_std, "N/A")

                mean_val = parse_mean_from_str(val_str)
                means[model] = mean_val

                latex_val = val_str.replace("±", r"\pm")

                if model != my_method:
                    p_str = row.get(col_pval, "")
                    sig_mark = get_significance_mark(p_str)
                    if sig_mark and latex_val != "N/A":
                        latex_val = f"${latex_val}{sig_mark}$"
                    elif latex_val != "N/A":
                        latex_val = f"${latex_val}$"
                else:
                    if latex_val != "N/A":
                        latex_val = f"${latex_val}$"

                row_strings[model] = latex_val

            valid_means = {k: v for k, v in means.items() if v != -np.inf}
            best_model = None
            if valid_means:
                if better_dir == 1:
                    best_model = max(valid_means, key=valid_means.get)
                else:
                    best_model = min(valid_means, key=valid_means.get)

            line_items = [metric_name.replace('_', r'\_')]  # 转义下划线

            for model in models_list:
                val_str = row_strings[model]
                if model == best_model and val_str != "N/A":
                    clean_content = val_str.replace('$', '')
                    line_items.append(r"$\mathbf{" + clean_content + r"}$")
                else:
                    line_items.append(val_str)

            latex_code.append(" & ".join(line_items) + r" \\")

        latex_code.append(r"\bottomrule")
        latex_code.append(r"\end{tabular}")
        latex_code.append(r"}")
        latex_code.append(r"\end{table}")

        print("\n".join(latex_code))
        print("\n")

generate_latex_tables(df, models_list, my_method)

reading data...
data loading finished, transferring to DataFrame...
Dataset: mcfarland
Dataset: zhaoSims
      Dataset                   Metric sc2Flow (Mean±SD) chemCPA (Mean±SD) chemCPA p-val biolord (Mean±SD) biolord p-val
0   mcfarland      test_r_squared_mean   0.9008 ± 0.0230   0.7852 ± 0.0599       0.0094*   0.9066 ± 0.0259        0.7164
1   mcfarland     test_e_distance_mean   1.3042 ± 0.3294   5.3037 ± 0.9861    3.943e-04*   4.1363 ± 0.4488    6.333e-06*
2   mcfarland            test_mmd_mean   0.0671 ± 0.0157   0.4490 ± 0.0690    1.516e-04*   0.6385 ± 0.0399    5.084e-07*
3   mcfarland   test_r_squared_mean_de   0.8568 ± 0.0562   0.7982 ± 0.0668        0.1724   0.8566 ± 0.0607        0.9954
4   mcfarland  test_e_distance_mean_de   0.8577 ± 0.2684   2.2555 ± 0.4391    6.234e-04*   1.9787 ± 0.3244    3.893e-04*
5   mcfarland         test_mmd_mean_de   0.0753 ± 0.0192   0.4949 ± 0.0550    1.808e-05*   0.5828 ± 0.0295    9.448e-09*
6    zhaoSims      test_r_squared_mean   0.8161 